# Task 1：实现 LIF 神经元

目标：模拟**一个神经元**接收恒定电流后，膜电位如何变化、何时发放脉冲。依次运行下方单元格即可。

## 1. 从作业公式到 `snn.Leaky`

作业的 Euler 更新公式是：

$$V_{t+\Delta t}=V_t+\frac{\Delta t}{\tau_m}\left[-(V_t-V_{\mathrm{rest}})+RI_t\right]$$

`snn.Leaky` 在发放重置前的基本更新公式是：

$$U_{t+1}=\beta U_t+X_t$$

令 $U_t=V_t-V_{\mathrm{rest}}$，即以静息电位为零点。将作业公式两侧减去 $V_{\mathrm{rest}}$，得到：

$$\begin{aligned}
U_{t+\Delta t}
&=U_t+\frac{\Delta t}{\tau_m}(-U_t+RI_t)\\
&=\left(1-\frac{\Delta t}{\tau_m}\right)U_t+\frac{\Delta t}{\tau_m}RI_t
\end{aligned}$$

与 `snn.Leaky` 逐项对照：

$$\boxed{\beta=1-\frac{\Delta t}{\tau_m}},\qquad \boxed{X_t=\frac{\Delta t}{\tau_m}RI_t}$$

因此使用上述参数时，两者的膜电位更新相同。例如 $\Delta t=0.1$ ms、$\tau_m=20$ ms、$R=10$ MΩ、$I=2$ nA 时，$\beta=0.995$，每步输入增量 $X=0.1$ mV。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch

## 2. 模拟一个神经元

使用作业的默认参数。当膜电位达到阈值时记录脉冲、立即重置；随后 2 ms 不应期内保持重置电位、不发放。`snn.Leaky` 默认用严格大于阈值，这里用 `at_threshold` 实现题目要求的“大于或等于”；不应期由计数器实现。

In [ ]:
# 作业建议的默认参数：电位 mV、时间 ms、电阻 MΩ、电流 nA
rest, reset, threshold = -65.0, -65.0, -50.0
dt, tau, resistance, current = 0.1, 20.0, 10.0, 2.0
refractory_ms, duration_ms = 2.0, 2000.0
voltage = rest  # 从静息电位开始

In [ ]:
def at_threshold(voltage):
    # 输入为相对阈值的电位：达到阈值（包括恰好等于）时发放。
    return (voltage >= 0).to(voltage.dtype)

In [ ]:
def simulate(current_na, duration_ms=duration_ms, dt=dt):
    neuron = snn.Leaky(
        beta=torch.tensor(1 - dt / tau, dtype=torch.float64),
        threshold=threshold - rest,
        reset_mechanism="zero",
        reset_delay=False,
        spike_grad=at_threshold,
    ).double()
    voltage = torch.tensor(reset - rest, dtype=torch.float64)
    drive = torch.tensor(dt / tau * resistance * current_na, dtype=torch.float64)
    remaining = 0
    voltages, spikes = [], []
    with torch.no_grad():
        for _ in range(round(duration_ms / dt)):
            if remaining > 0:  # 不应期：保持重置电位，不发放
                voltage = torch.tensor(reset - rest, dtype=torch.float64)
                spike = False
                remaining -= 1
            else:
                spike, voltage = neuron(drive, voltage)
                spike = bool(spike.item())
                if spike:
                    voltage = torch.tensor(reset - rest, dtype=torch.float64)
                    remaining = round(refractory_ms / dt)
            voltages.append(voltage.item() + rest)
            spikes.append(spike)
    return np.asarray(voltages), np.asarray(spikes)

## 3. 观察一次仿真

给神经元施加 2 nA 恒定电流，运行 2 秒。图中只展示前 200 ms，红虚线是发放阈值，圆点是脉冲发生的时刻。这里记录的是**每一步结束后的电位**，因此脉冲对应的电位已经重置。

In [ ]:
voltages, spikes = simulate(current_na=current, dt=dt)
time_ms = (np.arange(len(voltages)) + 1) * dt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_ms, voltages, label="Membrane potential")
ax.axhline(threshold, color="red", linestyle="--", label="Threshold")
ax.scatter(time_ms[spikes], voltages[spikes], s=8, label="Spike / reset")
ax.set(xlabel="Time (ms)", ylabel="Voltage (mV)")
ax.set_xlim(0, 200)
ax.legend()
plt.show()
print(
    f"脉冲数：{spikes.sum()}；平均发放率：{spikes.sum() / (len(spikes) * dt / 1000):.1f} Hz"
)

## 4. 用理论检查实现

默认参数下，临界恒定电流为 $(V_{th}-V_{rest})/R=1.5$ nA。低于临界值时电位达不到阈值；恰好等于临界值时电位渐近接近阈值，在有限时间内也不发放。将上面的 2 nA 改为 1 nA、1.5 nA 再运行，观察脉冲是否消失。

这里完成的是 Task 1。不同输入、系统对照实验和滑动发放率属于后续任务。